# Tutorial 01: Basic Harmonic Workflow - Sn_Zn in ZnO

This tutorial demonstrates the basic CarrierCapture workflow using a simple harmonic oscillator model.

We'll model the Sn_Zn defect in ZnO using symmetric harmonic potentials, following the example from the original CarrierCapture.jl package.

**Learning objectives:**
- Create harmonic potential energy surfaces
- Solve the 1D Schrödinger equation
- Calculate carrier capture coefficients
- Visualize configuration coordinate diagrams
- Generate Arrhenius plots

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt

from carriercapture.core.potential import Potential
from carriercapture.core.config_coord import ConfigCoordinate
from carriercapture.visualization import (
    plot_potential,
    plot_capture_coefficient,
    plot_configuration_coordinate,
)

# Configure plots
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 1. Create Harmonic Potentials

We model the initial (excited) and final (ground) states as harmonic oscillators.

For Sn_Zn in ZnO:
- Phonon frequency: ℏω = 8 meV (typical for ZnO)
- Configuration coordinate shift: ΔQ = 10.5 amu^0.5·Å
- Energy difference: ΔE = 0.5 eV

In [ ]:
# Parameters
hw = 0.008  # Phonon energy (eV)
dQ = 10.5   # Configuration coordinate shift (amu^0.5·Å)
dE = 0.5    # Energy difference (eV)

# Create initial state (excited) - centered at Q=0, shifted up by dE
pot_initial = Potential.from_harmonic(
    hw=hw,
    Q0=0.0,
    E0=dE,
    Q_range=(-20, 20),
    npoints=5000,
)

# Create final state (ground) - centered at Q=dQ
pot_final = Potential.from_harmonic(
    hw=hw,
    Q0=dQ,
    E0=0.0,
    Q_range=(-20, 20),
    npoints=5000,
)

print(f"Created harmonic potentials:")
print(f"  Initial: Q₀={pot_initial.Q0:.1f}, E₀={pot_initial.E0:.3f} eV")
print(f"  Final:   Q₀={pot_final.Q0:.1f}, E₀={pot_final.E0:.3f} eV")

## 2. Solve the Schrödinger Equation

We solve the 1D time-independent Schrödinger equation to find phonon states (eigenvalues and eigenvectors).

For harmonic oscillators, we expect:
- Eigenvalues: E_n = ℏω(n + 1/2)
- Equally spaced energy levels

In [ ]:
# Solve for phonon states
pot_initial.solve(nev=180)  # More states for initial (hotter)
pot_final.solve(nev=60)     # Fewer states for final

print(f"\nSolved Schrödinger equation:")
print(f"  Initial: {len(pot_initial.eigenvalues)} eigenvalues")
print(f"  Final:   {len(pot_final.eigenvalues)} eigenvalues")
print(f"\nFirst 5 eigenvalues (initial state):")
for i in range(5):
    E_n = pot_initial.eigenvalues[i]
    E_theory = hw * (i + 0.5) + dE
    print(f"  n={i}: E={E_n:.6f} eV (theory: {E_theory:.6f} eV, error: {abs(E_n-E_theory):.2e})")

## 3. Visualize Potentials

Let's plot the potential energy surfaces with phonon states overlaid.

In [ ]:
# Plot initial state
fig1 = plot_potential(pot_initial, show_wavefunctions=True, n_wf=5)
fig1.show()

# Plot final state
fig2 = plot_potential(pot_final, show_wavefunctions=True, n_wf=5)
fig2.show()

## 4. Configuration Coordinate Diagram

Plot both potentials together to visualize the carrier capture process.

In [ ]:
# Create ConfigCoordinate object
W = 0.068  # Electron-phonon coupling (eV)
cc = ConfigCoordinate(
    pot_i=pot_initial,
    pot_f=pot_final,
    W=W,
    degeneracy=1,
)

# Plot configuration coordinate diagram
fig3 = plot_configuration_coordinate(cc)
fig3.show()

## 5. Calculate Carrier Capture Coefficient

The capture coefficient C(T) describes the rate of nonradiative carrier capture.

Key steps:
1. Calculate overlap matrix elements between initial and final states
2. Sum over thermally occupied states
3. Apply Fermi's golden rule

In [ ]:
# Calculate overlap matrix
Q0 = dQ / 2  # Crossing point (midpoint)
cc.calculate_overlap(
    Q0=Q0,
    cutoff=0.25,  # Energy window for overlaps (eV)
    sigma=0.01,   # Gaussian broadening (eV)
)

print(f"\nOverlap calculation complete:")
print(f"  Overlap matrix shape: {cc.overlap_matrix.shape}")
print(f"  Non-zero overlaps: {np.sum(cc.overlap_matrix != 0)}")

# Calculate capture coefficient over temperature range
temperature = np.linspace(100, 500, 50)  # 100-500 K
volume = 1e-21  # Supercell volume (cm³)

cc.calculate_capture_coefficient(
    volume=volume,
    temperature=temperature,
)

print(f"\nCapture coefficient at selected temperatures:")
for i, T in enumerate([100, 200, 300, 400, 500]):
    idx = np.argmin(np.abs(temperature - T))
    C = cc.capture_coefficient[idx]
    print(f"  T={T:3d} K: C={C:.3e} cm³/s")

## 6. Arrhenius Plot

Plot log(C) vs 1000/T to see the temperature dependence.

In [ ]:
fig4 = plot_capture_coefficient(cc)
fig4.show()

## 7. Physical Interpretation

The capture coefficient shows:

1. **Temperature dependence**: C(T) typically increases with temperature due to:
   - Greater thermal population of higher phonon states
   - Enhanced overlap between initial and final states

2. **Magnitude**: For Sn_Zn, C ~ 10⁻¹¹ cm³/s at room temperature
   - Fast capture process (lifetime ~ ns for typical carrier densities)

3. **Activation energy**: The slope in the Arrhenius plot reveals the activation barrier
   - Related to the crossing point energy

## Summary

In this tutorial, we:
- ✓ Created harmonic potentials for initial and final states
- ✓ Solved the Schrödinger equation (validated against analytical solution)
- ✓ Calculated overlap matrix elements
- ✓ Computed carrier capture coefficients vs temperature
- ✓ Visualized results with publication-quality figures

**Next steps:**
- Try modifying ΔQ, ΔE, ℏω to see their effect
- Explore Tutorial 02 for anharmonic potentials with real DFT data
- Use Tutorial 03 for high-throughput parameter scanning

## Exercises

1. **Effect of ΔQ**: Change `dQ` from 5 to 15 and observe how C(T) changes
2. **Effect of ΔE**: Vary `dE` from 0.1 to 1.0 eV
3. **Phonon frequency**: Try different ℏω values (4, 8, 12 meV)
4. **Convergence**: How many eigenvalues (nev) are needed for converged results?

In [ ]:
# Exercise 1: Scan over dQ values
dQ_values = [5, 7.5, 10, 12.5, 15]
C_300K = []

for dQ_test in dQ_values:
    # Create potentials
    pot_i_test = Potential.from_harmonic(hw=hw, Q0=0.0, E0=dE, Q_range=(-20, 20), npoints=3000)
    pot_f_test = Potential.from_harmonic(hw=hw, Q0=dQ_test, E0=0.0, Q_range=(-20, 20), npoints=3000)
    
    # Solve
    pot_i_test.solve(nev=180)
    pot_f_test.solve(nev=60)
    
    # Calculate
    cc_test = ConfigCoordinate(pot_i=pot_i_test, pot_f=pot_f_test, W=W)
    cc_test.calculate_overlap(Q0=dQ_test/2, cutoff=0.25, sigma=0.01)
    cc_test.calculate_capture_coefficient(volume=volume, temperature=np.array([300.0]))
    
    C_300K.append(cc_test.capture_coefficient[0])

# Plot results
plt.figure(figsize=(8, 5))
plt.semilogy(dQ_values, C_300K, 'o-', linewidth=2, markersize=8)
plt.xlabel('ΔQ (amu$^{0.5}$·Å)', fontsize=14)
plt.ylabel('C(300K) (cm³/s)', fontsize=14)
plt.title('Effect of Configuration Coordinate Shift', fontsize=16)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nCapture coefficient at 300K vs ΔQ:")
for dQ_val, C_val in zip(dQ_values, C_300K):
    print(f"  ΔQ={dQ_val:5.1f}: C={C_val:.3e} cm³/s")